# PySpark interview exercises

Run the cells from top to bottom.

In [1]:
import os

# VS Code notebook kernels do not always inherit ~/.zshrc.
java_home = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
os.environ["JAVA_HOME"] = java_home

os.environ["PATH"] = f"{java_home}/bin:{os.environ['PATH']}"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("MySparkApp")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/10 16:15:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Sample events

In [2]:
df = spark.createDataFrame(
    [
        {
            "event_id": 1,
            "user_id": 10,
            "event_type": "booking",
            "event_timestamp": "2026-01-01 10:00:00",
            "domain": "example.com",
            "country": "ES",
        },
        {
            "event_id": 2,
            "user_id": 11,
            "event_type": "click",
            "event_timestamp": "2026-01-01 11:00:00",
            "domain": "example.com",
            "country": "UK",
        },
        {
            "event_id":3,
            "user_id": 11,
            "event_type": "click",
            "event_timestamp": "2026-01-02 11:00:00",
            "domain": "example.com",
            "country": "UK",
        },
        {
            "event_id": 1,
            "user_id": 11,
            "event_type": "click",
            "event_timestamp": "2026-01-02 11:00:00",
            "domain": "example.com",
            "country": "US",
        },
    ]
)

df.show()

+-------+-----------+--------+-------------------+----------+-------+
|country|     domain|event_id|    event_timestamp|event_type|user_id|
+-------+-----------+--------+-------------------+----------+-------+
|     ES|example.com|       1|2026-01-01 10:00:00|   booking|     10|
|     UK|example.com|       2|2026-01-01 11:00:00|     click|     11|
|     UK|example.com|       3|2026-01-02 11:00:00|     click|     11|
|     US|example.com|       1|2026-01-02 11:00:00|     click|     11|
+-------+-----------+--------+-------------------+----------+-------+



## Exercise solutions

### Filter
Return only booking events

In [3]:
df.filter(df.event_type == "booking").show()

+-------+-----------+--------+-------------------+----------+-------+
|country|     domain|event_id|    event_timestamp|event_type|user_id|
+-------+-----------+--------+-------------------+----------+-------+
|     ES|example.com|       1|2026-01-01 10:00:00|   booking|     10|
+-------+-----------+--------+-------------------+----------+-------+



In [4]:
df.where(df.event_type == "booking").show()

+-------+-----------+--------+-------------------+----------+-------+
|country|     domain|event_id|    event_timestamp|event_type|user_id|
+-------+-----------+--------+-------------------+----------+-------+
|     ES|example.com|       1|2026-01-01 10:00:00|   booking|     10|
+-------+-----------+--------+-------------------+----------+-------+



### Aggregation
count events by country and event_type

In [5]:
# when we groupby in pyspark we usually order a shuffle bc
# we pull data with the same key as per the grouping keys into the same partition
# to then be able to perform the aggregation operation
df.groupBy(["country", "event_type"]).count().show()

+-------+----------+-----+
|country|event_type|count|
+-------+----------+-----+
|     ES|   booking|    1|
|     UK|     click|    2|
|     US|     click|    1|
+-------+----------+-----+



26/09/10 21:53:51 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 935005 ms exceeds timeout 120000 ms
26/09/10 21:53:51 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/10 21:53:57 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

a very common issue is that during a groupBy a Spark task might take far longer than all the others. The Spark task taking longer is also the partition containing more data than the rest.

This could be several things:
1. Data partitioning key not in line with groupBy key.
2. If one task is much slower than the rest and also far bigger than the rest this might point to a **data skew**. This happens when for one of the keys is massively overrepresented in the data in comparison to the rest:
```md
US, search  -> 1.8 billion rows
ES, search  -> 120 million rows
FR, click   -> 40 million rows
```
in this case, the partitions won't be balanced and the task handling this partition becomes a straggler.

**How to mitigate this?**
- rethink partitioning keys
- salting hot keys
- pre-aggregating before shuffle
- using skew aware join/aggregation techniques

### deduplication
keep latest row for each `event_id`.

In [48]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# group by event id, sort by timestamp, keep only first
window = (
    Window
    .partitionBy("event_id")
    .orderBy(F.col("event_timestamp").desc())
)

In [49]:
ranked = df.withColumn(
    "row_num",
    F.row_number().over(window)
)

In [50]:
ranked.show()

+-------+-----------+--------+-------------------+----------+-------+-------+
|country|     domain|event_id|    event_timestamp|event_type|user_id|row_num|
+-------+-----------+--------+-------------------+----------+-------+-------+
|     US|example.com|       1|2026-01-02 11:00:00|     click|     11|      1|
|     ES|example.com|       1|2026-01-01 10:00:00|   booking|     10|      2|
|     UK|example.com|       2|2026-01-01 11:00:00|     click|     11|      1|
|     UK|example.com|       3|2026-01-02 11:00:00|     click|     11|      1|
+-------+-----------+--------+-------------------+----------+-------+-------+



In [51]:
result = (
    ranked
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

In [ ]:
result.show()

+-------+-----------+--------+-------------------+----------+-------+
|country|     domain|event_id|    event_timestamp|event_type|user_id|
+-------+-----------+--------+-------------------+----------+-------+
|     US|example.com|       1|2026-01-02 11:00:00|     click|     11|
|     UK|example.com|       2|2026-01-01 11:00:00|     click|     11|
|     UK|example.com|       3|2026-01-02 11:00:00|     click|     11|
+-------+-----------+--------+-------------------+----------+-------+



26/09/04 15:31:26 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 944620 ms exceeds timeout 120000 ms
26/09/04 15:31:26 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/04 15:31:30 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:34)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.stora

### top N
return top 3 event types per country

### join


## Cleanup

Run this after finishing the exercises.

In [37]:
spark.stop()